# Árbol de Decisión (Decision Tree)

## ¿Cuál es el objetivo del Árbol de Decisión?

Un **Árbol de Decisión** es un algoritmo de **clasificación** (y también regresión) que aprende una serie de **reglas de decisión** tipo "si-entonces" a partir de los datos, organizadas en una estructura de árbol.

La idea es intuitiva: para clasificar un nuevo punto, el árbol hace una **pregunta** sobre una feature en cada nodo ("¿es $x_1 \leq 3.5$?") y, según la respuesta, sigue por la rama izquierda o derecha hasta llegar a una **hoja** que contiene la clase predicha.

El objetivo del algoritmo es **construir el árbol** encontrando, en cada nodo, la **mejor pregunta** que divida los datos de la forma más "pura" posible. Formalmente, buscamos la feature $j$ y el umbral $t$ que **maximizan la ganancia de información**:

$$
\text{IG}(y, x_j, t) = H(y) - \left[\frac{n_{\text{izq}}}{n} H(y_{\text{izq}}) + \frac{n_{\text{der}}}{n} H(y_{\text{der}})\right]
$$

Donde:

- $H(y)$: **entropía** del nodo padre (mide la impureza).
- $H(y_{\text{izq}})$, $H(y_{\text{der}})$: entropía de los nodos hijos.
- $n_{\text{izq}}$, $n_{\text{der}}$: número de muestras en cada hijo.
- $n$: número total de muestras en el nodo.

**Ventajas clave:**
- **Interpretable**: puedes visualizar y entender cada decisión.
- **No requiere normalización** de features.
- Maneja features **numéricas y categóricas**.
- Es la base de algoritmos poderosos como **Random Forest** y **Gradient Boosting**.

**Supongamos un conjunto de datos simples**

Queremos clasificar si una persona juega tenis basándonos en el clima:

| Temperatura (°C) | Humedad (%) | Juega tenis |
|------------------|-------------|-------------|
| 30               | 85          | No (0)      |
| 28               | 90          | No (0)      |
| 22               | 70          | Sí (1)      |
| 25               | 60          | Sí (1)      |
| 20               | 80          | Sí (1)      |
| 35               | 75          | No (0)      |
| 18               | 65          | Sí (1)      |
| 32               | 95          | No (0)      |

El árbol intentará encontrar umbrales como "¿Temperatura $\leq$ 27°C?" que separen las clases.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Datos de ejemplo
temp = np.array([30, 28, 22, 25, 20, 35, 18, 32])
humedad = np.array([85, 90, 70, 60, 80, 75, 65, 95])
juega = np.array([0, 0, 1, 1, 1, 0, 1, 0])

plt.figure(figsize=(8, 6))
plt.scatter(temp[juega == 0], humedad[juega == 0], color='red', s=100,
            label='No juega (0)', edgecolors='black', marker='X')
plt.scatter(temp[juega == 1], humedad[juega == 1], color='green', s=100,
            label='Sí juega (1)', edgecolors='black')

# Dibujar un posible split
plt.axvline(x=27, color='blue', linestyle='--', linewidth=2, alpha=0.7,
            label='Split: Temp ≤ 27°C')

for i, (t, h) in enumerate(zip(temp, humedad)):
    plt.annotate(f'  ({t}, {h})', (t, h), fontsize=8)

plt.xlabel('Temperatura (°C)')
plt.ylabel('Humedad (%)')
plt.title('¿Jugar tenis? El árbol busca los mejores splits')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## ¿Cómo construye el árbol sus reglas?

El árbol se construye de forma **recursiva** y **greedy** (glotona):

1. En cada nodo, **probar todas las features** y todos los umbrales posibles.
2. Elegir el split que produce la **mayor ganancia de información**.
3. Dividir los datos en dos grupos (izquierda y derecha).
4. Repetir recursivamente en cada grupo.
5. Detenerse cuando se cumple algún **criterio de parada** (profundidad máxima, nodo puro, o pocas muestras).

La herramienta clave para medir "qué tan bueno" es un split es la **entropía**.

## Demostración Matemática

### Paso 1: Entropía

La **entropía** mide el desorden o impureza de un conjunto de datos. Para un nodo con $C$ clases:

$$
H(y) = -\sum_{c=1}^{C} p_c \log_2(p_c)
$$

Donde $p_c = \frac{n_c}{n}$ es la proporción de muestras de la clase $c$.

**Propiedades:**

- $H = 0$: el nodo es **puro** (todas las muestras son de la misma clase). Es el caso ideal.
- $H = 1$ (para 2 clases): máxima impureza (50% de cada clase). Es el peor caso.

**Intuición:** la entropía responde a la pregunta "¿cuánta **incertidumbre** hay sobre la clase de una muestra tomada al azar de este nodo?"

Ejemplo:
- Si un nodo tiene 10 muestras, todas de clase 1: $H = -1 \cdot \log_2(1) = 0$ (certeza total).
- Si un nodo tiene 5 de clase 0 y 5 de clase 1: $H = -0.5 \cdot \log_2(0.5) - 0.5 \cdot \log_2(0.5) = 1$ (máxima incertidumbre).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

p = np.linspace(0.001, 0.999, 200)
entropy = -p * np.log2(p) - (1 - p) * np.log2(1 - p)

plt.figure(figsize=(8, 5))
plt.plot(p, entropy, 'b-', linewidth=2)
plt.scatter([0.5], [1.0], color='red', s=100, zorder=5)
plt.annotate('Máx. impureza\nH = 1.0', xy=(0.5, 1.0), xytext=(0.65, 0.85),
             fontsize=11, arrowprops=dict(arrowstyle='->', color='red'),
             bbox=dict(boxstyle='round', facecolor='lightsalmon', alpha=0.7))

plt.annotate('Nodo puro\n(clase 0)', xy=(0.01, 0.08), fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
plt.annotate('Nodo puro\n(clase 1)', xy=(0.85, 0.08), fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.xlabel('Proporción de clase 1 ($p$)')
plt.ylabel('Entropía $H$')
plt.title('Entropía binaria: $H = -p\log_2(p) - (1-p)\log_2(1-p)$')
plt.grid(True, alpha=0.3)
plt.show()

### Paso 2: Ganancia de Información (Information Gain)

La **ganancia de información** mide cuánta entropía **reducimos** al dividir los datos con una feature $x_j$ y un umbral $t$:

$$
\text{IG}(y, x_j, t) = H(y) - H(y | x_j, t)
$$

Donde la entropía condicional es el **promedio ponderado** de la entropía de los hijos:

$$
H(y | x_j, t) = \frac{n_{\text{izq}}}{n} H(y_{\text{izq}}) + \frac{n_{\text{der}}}{n} H(y_{\text{der}})
$$

Con:
- $y_{\text{izq}} = \{y_i : x_{ij} \leq t\}$ (muestras que van a la izquierda).
- $y_{\text{der}} = \{y_i : x_{ij} > t\}$ (muestras que van a la derecha).

**Propiedades:**

- $\text{IG} \geq 0$ siempre (un split nunca puede empeorar la impureza total).
- $\text{IG} = 0$ cuando el split no aporta nada (la distribución de clases es la misma en ambos hijos).
- $\text{IG}$ es máximo cuando el split genera **hijos puros**.

El algoritmo elige el split $(j^*, t^*)$ que maximiza la ganancia:

$$
(j^*, t^*) = \arg\max_{j, t} \ \text{IG}(y, x_j, t)
$$

### Paso 3: Construcción recursiva del árbol

El algoritmo construye el árbol de forma **top-down** y **greedy**:

```
función CRECER_ÁRBOL(X, y, profundidad):
    si criterio_de_parada:
        retornar HOJA(clase_más_común(y))
    
    (j*, t*) = mejor_split(X, y)
    
    izq = muestras donde X[:, j*] ≤ t*
    der = muestras donde X[:, j*] > t*
    
    hijo_izq = CRECER_ÁRBOL(X[izq], y[izq], profundidad + 1)
    hijo_der = CRECER_ÁRBOL(X[der], y[der], profundidad + 1)
    
    retornar NODO(j*, t*, hijo_izq, hijo_der)
```

**Criterios de parada:**

1. **Profundidad máxima**: $\text{depth} \geq \text{max\_depth}$
2. **Nodo puro**: solo una clase presente ($|\text{unique}(y)| = 1$)
3. **Pocas muestras**: $n < \text{min\_samples\_split}$

Cuando se alcanza un criterio de parada, se crea una **hoja** cuyo valor es la **clase más común** (moda) entre las muestras que llegan a ese nodo.

### Paso 4: Predicción

Para predecir la clase de un nuevo punto $\mathbf{x}$, simplemente **recorremos el árbol** desde la raíz:

1. En cada nodo, evaluar la condición: $x_{j} \leq t$?
2. Si **sí**, ir al hijo izquierdo.
3. Si **no**, ir al hijo derecho.
4. Repetir hasta llegar a una **hoja**.
5. Retornar el valor de la hoja.

La complejidad de predicción es $O(\text{profundidad})$, que es muy eficiente.

### Hiperparámetros clave

- **max_depth**: profundidad máxima del árbol. Limita la complejidad del modelo.
  - Bajo: subajuste (reglas demasiado simples).
  - Alto: sobreajuste (memoriza los datos de entrenamiento).
- **min_samples_split**: mínimo de muestras para dividir un nodo.
- **n_feats**: número de features a considerar en cada split (útil para Random Forest, donde se usa un subconjunto aleatorio).

### Ejemplo numérico

Usemos nuestros datos de tenis para calcular el mejor primer split.

| Temperatura (°C) | Humedad (%) | Juega tenis |
|------------------|-------------|-------------|
| 30               | 85          | 0           |
| 28               | 90          | 0           |
| 22               | 70          | 1           |
| 25               | 60          | 1           |
| 20               | 80          | 1           |
| 35               | 75          | 0           |
| 18               | 65          | 1           |
| 32               | 95          | 0           |

**Entropía del nodo raíz:**

Clases: 4 de clase 0, 4 de clase 1 → $p_0 = 0.5$, $p_1 = 0.5$

$$
H(y) = -0.5 \log_2(0.5) - 0.5 \log_2(0.5) = 0.5 + 0.5 = 1.0
$$

**Probemos el split: Temperatura $\leq$ 27**

- **Izquierda** ($T \leq 27$): $\{22, 25, 20, 18\}$ → clases $\{1, 1, 1, 1\}$ → $H = 0$
- **Derecha** ($T > 27$): $\{30, 28, 35, 32\}$ → clases $\{0, 0, 0, 0\}$ → $H = 0$

$$
\text{IG} = 1.0 - \left[\frac{4}{8} \cdot 0 + \frac{4}{8} \cdot 0\right] = 1.0 - 0 = 1.0
$$

¡Ganancia máxima de 1.0! Este split separa **perfectamente** las dos clases.

**Probemos otro split: Humedad $\leq$ 80**

- **Izquierda** ($H \leq 80$): $\{70, 60, 80, 75, 65\}$ → clases $\{1, 1, 1, 0, 1\}$ → $p_1 = 4/5$, $p_0 = 1/5$

$$H_{\text{izq}} = -\frac{4}{5}\log_2\frac{4}{5} - \frac{1}{5}\log_2\frac{1}{5} \approx 0.722$$

- **Derecha** ($H > 80$): $\{85, 90, 95\}$ → clases $\{0, 0, 0\}$ → $H_{\text{der}} = 0$

$$
\text{IG} = 1.0 - \left[\frac{5}{8} \cdot 0.722 + \frac{3}{8} \cdot 0\right] = 1.0 - 0.451 = 0.549
$$

Ganancia de 0.549, que es buena pero menor que el split por temperatura.

$$
\boxed{\text{Mejor split: Temperatura} \leq 27 \text{ con IG} = 1.0}
$$

In [ ]:
import numpy as np

def entropy(y):
    hist = np.bincount(y)
    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])

# Datos
X = np.array([[30, 85], [28, 90], [22, 70], [25, 60],
              [20, 80], [35, 75], [18, 65], [32, 95]])
y = np.array([0, 0, 1, 1, 1, 0, 1, 0])
feature_names = ['Temperatura', 'Humedad']

print("=== Ejemplo numérico: Decision Tree paso a paso ===")
print(f"Entropía del nodo raíz: H = {entropy(y):.4f}\n")

# Probar todos los splits posibles
print("Evaluando todos los splits posibles:")
print("-" * 70)

best_gain = -1
best_feat = None
best_thresh = None

for feat_idx in range(2):
    thresholds = np.unique(X[:, feat_idx])
    for thresh in thresholds:
        left_mask = X[:, feat_idx] <= thresh
        right_mask = ~left_mask

        if left_mask.sum() == 0 or right_mask.sum() == 0:
            continue

        e_left = entropy(y[left_mask])
        e_right = entropy(y[right_mask])
        n = len(y)
        child_entropy = (left_mask.sum() / n) * e_left + (right_mask.sum() / n) * e_right
        ig = entropy(y) - child_entropy

        marker = " ← MEJOR" if ig > best_gain else ""
        if ig > best_gain:
            best_gain = ig
            best_feat = feat_idx
            best_thresh = thresh

        print(f"  {feature_names[feat_idx]:12s} ≤ {thresh:5.0f}  │  "
              f"Izq: {left_mask.sum()} muestras (H={e_left:.3f})  │  "
              f"Der: {right_mask.sum()} muestras (H={e_right:.3f})  │  "
              f"IG = {ig:.4f}{marker}")

print("-" * 70)
print(f"\nMejor split: {feature_names[best_feat]} ≤ {best_thresh} con IG = {best_gain:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Visualizar el árbol resultante como diagrama
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

# Nodo raíz
root = mpatches.FancyBboxPatch((3.5, 4.5), 3, 1, boxstyle="round,pad=0.1",
                                facecolor="lightyellow", edgecolor="black", linewidth=2)
ax.add_patch(root)
ax.text(5, 5.1, 'Temperatura ≤ 27?', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(5, 4.7, 'H = 1.0 | 8 muestras', ha='center', va='center', fontsize=9, color='gray')

# Hijo izquierdo (Sí)
left = mpatches.FancyBboxPatch((0.5, 1.5), 3, 1, boxstyle="round,pad=0.1",
                                facecolor="lightgreen", edgecolor="black", linewidth=2)
ax.add_patch(left)
ax.text(2, 2.1, 'Clase: Sí juega (1)', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(2, 1.7, 'H = 0.0 | 4 muestras', ha='center', va='center', fontsize=9, color='gray')

# Hijo derecho (No)
right = mpatches.FancyBboxPatch((6.5, 1.5), 3, 1, boxstyle="round,pad=0.1",
                                 facecolor="lightsalmon", edgecolor="black", linewidth=2)
ax.add_patch(right)
ax.text(8, 2.1, 'Clase: No juega (0)', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(8, 1.7, 'H = 0.0 | 4 muestras', ha='center', va='center', fontsize=9, color='gray')

# Flechas
ax.annotate('', xy=(2, 2.5), xytext=(4.2, 4.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='green'))
ax.annotate('', xy=(8, 2.5), xytext=(5.8, 4.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='red'))

ax.text(2.5, 3.7, 'Sí (≤ 27)', fontsize=11, color='green', fontweight='bold')
ax.text(6.5, 3.7, 'No (> 27)', fontsize=11, color='red', fontweight='bold')

plt.title('Árbol de Decisión resultante', fontsize=14, fontweight='bold', pad=20)
plt.show()

## Implementación con Código

### Funciones

In [ ]:
from collections import Counter
import numpy as np

def entropy(y):
    """Calcula la entropía de un conjunto de etiquetas"""
    hist = np.bincount(y)
    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])

def information_gain(y, X_column, threshold):
    """Calcula la ganancia de información de un split"""
    parent_entropy = entropy(y)

    left_idxs = np.argwhere(X_column <= threshold).flatten()
    right_idxs = np.argwhere(X_column > threshold).flatten()

    if len(left_idxs) == 0 or len(right_idxs) == 0:
        return 0

    n = len(y)
    n_l, n_r = len(left_idxs), len(right_idxs)
    e_l, e_r = entropy(y[left_idxs]), entropy(y[right_idxs])
    child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

    return parent_entropy - child_entropy

def best_split(X, y):
    """Encuentra el mejor split (feature + umbral)"""
    best_gain = -1
    best_feat, best_thresh = None, None

    for feat_idx in range(X.shape[1]):
        thresholds = np.unique(X[:, feat_idx])
        for threshold in thresholds:
            gain = information_gain(y, X[:, feat_idx], threshold)
            if gain > best_gain:
                best_gain = gain
                best_feat = feat_idx
                best_thresh = threshold

    return best_feat, best_thresh, best_gain

In [ ]:
# Probar con los datos de tenis
X = np.array([[30, 85], [28, 90], [22, 70], [25, 60],
              [20, 80], [35, 75], [18, 65], [32, 95]])
y = np.array([0, 0, 1, 1, 1, 0, 1, 0])

feat, thresh, gain = best_split(X, y)
names = ['Temperatura', 'Humedad']
print(f"Mejor split: {names[feat]} ≤ {thresh} (IG = {gain:.4f})")

### Clase

In [ ]:
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


def entropy(y):
    hist = np.bincount(y)
    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])


class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None


class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=100, n_feats=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_feats = n_feats
        self.root = None

    def fit(self, X, y):
        """Construye el árbol recursivamente"""
        self.n_feats = X.shape[1] if not self.n_feats else min(self.n_feats, X.shape[1])
        self.root = self._grow_tree(X, y)

    def predict(self, X):
        """Predice la clase de cada muestra recorriendo el árbol"""
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def score(self, X, y):
        """Calcula la accuracy del modelo"""
        predictions = self.predict(X)
        return np.sum(predictions == y) / len(y)

    def _grow_tree(self, X, y, depth=0):
        """Construye el árbol recursivamente"""
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Criterios de parada
        if (depth >= self.max_depth
                or n_labels == 1
                or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feat_idxs = np.random.choice(n_features, self.n_feats, replace=False)

        # Seleccionar greedy el mejor split por información gain
        best_feat, best_thresh = self._best_criteria(X, y, feat_idxs)

        # Crecer los hijos
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)

    def _best_criteria(self, X, y, feat_idxs):
        """Encuentra el mejor split entre las features dadas"""
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold

        return split_idx, split_thresh

    def _information_gain(self, y, X_column, split_thresh):
        """Calcula la ganancia de información"""
        parent_entropy = entropy(y)

        left_idxs, right_idxs = self._split(X_column, split_thresh)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = entropy(y[left_idxs]), entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_entropy - child_entropy

    def _split(self, X_column, split_thresh):
        """Divide los datos según el umbral"""
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _traverse_tree(self, x, node):
        """Recorre el árbol para predecir un punto"""
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def _most_common_label(self, y):
        """Retorna la clase más frecuente"""
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def print_tree(self, node=None, depth=0, prefix="Raíz"):
        """Imprime el árbol de forma legible"""
        if node is None:
            node = self.root

        indent = "  " * depth
        if node.is_leaf_node():
            print(f"{indent}{prefix} → Clase {node.value}")
        else:
            print(f"{indent}{prefix}: Feature {node.feature} ≤ {node.threshold:.2f}")
            self.print_tree(node.left, depth + 1, "Sí")
            self.print_tree(node.right, depth + 1, "No")

    def plot_decision_boundary(self, X, y, title="Frontera de decisión - Decision Tree",
                               feature_indices=(0, 1)):
        """Grafica la frontera de decisión usando dos features"""
        i, j = feature_indices
        X_2d = X[:, [i, j]]

        temp_tree = DecisionTree(
            min_samples_split=self.min_samples_split,
            max_depth=self.max_depth
        )
        temp_tree.fit(X_2d, y)

        h = 0.1
        x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
        y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                             np.arange(y_min, y_max, h))

        Z = temp_tree.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)

        n_classes = len(np.unique(y))
        light_colors = ['#FFAAAA', '#AAFFAA', '#AAAAFF'][:n_classes]
        bold_colors = ['#FF0000', '#00FF00', '#0000FF'][:n_classes]
        cmap_light = ListedColormap(light_colors)
        cmap_bold = ListedColormap(bold_colors)

        plt.figure(figsize=(8, 6))
        plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
        plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap=cmap_bold, edgecolors='k', s=60)
        plt.xlabel(f'Feature {i}')
        plt.ylabel(f'Feature {j}')
        plt.title(f'{title} (max_depth={self.max_depth})')
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split

# Cargar datos
data = datasets.load_breast_cancer()
X, y = data.data, data.target

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1234
)

# Crear y entrenar modelo
model = DecisionTree(max_depth=10)
model.fit(X_train, y_train)

# Evaluar
acc = model.score(X_test, y_test)
print(f"Decision Tree accuracy: {acc:.4f}")

# Imprimir estructura del árbol (primeros niveles)
print("\nEstructura del árbol:")
model.print_tree()

In [ ]:
# Visualizar frontera de decisión con Iris (más visual en 2D)
iris = datasets.load_iris()
X_iris, y_iris = iris.data, iris.target
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=1234
)

model_iris = DecisionTree(max_depth=5)
model_iris.fit(X_train_iris, y_train_iris)
acc_iris = model_iris.score(X_test_iris, y_test_iris)
print(f"Decision Tree accuracy en Iris: {acc_iris:.4f}")

model_iris.plot_decision_boundary(X_train_iris, y_train_iris)

In [ ]:
# Comparar diferentes profundidades máximas
print("Efecto de max_depth en accuracy:\n")

depths = [1, 2, 3, 5, 10, 20, 50, 100]
train_accs = []
test_accs = []

for d in depths:
    model_d = DecisionTree(max_depth=d)
    model_d.fit(X_train, y_train)
    train_acc = model_d.score(X_train, y_train)
    test_acc = model_d.score(X_test, y_test)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    print(f"  max_depth = {d:3d}: train_acc = {train_acc:.4f}, test_acc = {test_acc:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(depths, train_accs, 'bo-', label='Train accuracy')
plt.plot(depths, test_accs, 'ro-', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Efecto de la profundidad máxima (sobreajuste vs subajuste)')
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()